# 🔬 Experiment Explorer (K-Fold)
Carga todos los experimentos de `RESULTS_PATH`, detecta automáticamente los folds (`fold_0`, `fold_1`, …),
agrega métricas con **media ± desviación estándar** entre folds, y ordena por la métrica elegida.

Las métricas provienen siempre de **`best_eval.csv`** (mejor época de validación por fold),
con fallback a `val_metrics.csv` si no existe.

**Toda la lógica de carga y parsing es compartida con `test.py` vía `utils/test_utils.py`.**

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN  ← edita aquí
# ─────────────────────────────────────────────
RESULTS_PATH  = '/workspace/results/experiment2/experiments'  # carpeta raíz con todos los experimentos
RANK_METRIC   = 'balanced_accuracy'            # métrica de ranking: 'auroc', 'f1_macro', 'balanced_accuracy', 'mcc', ...
TOP_N         = 5                  # filas por subgrupo (tarea × encoder × freeze × mil)
TOP_N_GLOBAL  = 5                 # filas en el ranking global final
TASK_FILTER   = None               # None → todas | p.ej. 'er', 'molecular_subtype', ...
ASCENDING     = False              # False → mayor es mejor (AUC, F1) | True → menor es mejor

METRIC_COLS = [
    'balanced_accuracy',
    'accuracy',
    'kappa',
    'f1_macro',
    'auprc_macro',
    'auroc', 'auroc_macro',
]

# Métricas para curvas de evolución
PLOT_METRICS = ['loss', 'balanced_accuracy', 'auroc', 'auroc_macro', 'f1_macro']

EXTRA_RUN = None   # ← pon aquí un run_name concreto para verlo además de los mejores, o None para omitir


In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML

# ── Importar funciones compartidas con test.py ────────────────────────────────
import sys
# Ajusta esta ruta si el notebook no está en la raíz del proyecto
sys.path.insert(0, os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__)))

from utils.test_utils import (
    collect_all_runs_df,   # construye el DataFrame maestro
    get_fold_dirs,         # para curvas de evolución
    FOLD_RE,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

print('Imports OK ✓')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# UTILIDADES LOCALES DE CARGA DE CSVs  (solo para curvas de evolución)
# ─────────────────────────────────────────────────────────────────────────────

def load_csv(directory, split):
    """Carga <directory>/<split>_metrics.csv o <directory>/<split>.csv,
    quitando classification_report si existe."""
    for fname in (f'{split}_metrics.csv', f'{split}.csv'):
        p = os.path.join(directory, fname)
        if os.path.isfile(p):
            df = pd.read_csv(p)
            return df.drop(columns=['classification_report'], errors='ignore')
    return None

def load_fold_epoch_dfs(run_dir, split):
    """Carga el CSV de 'split' para cada fold_i y devuelve lista de DataFrames."""
    fold_dirs = get_fold_dirs(run_dir)
    dfs = []
    for fd in fold_dirs:
        df = load_csv(fd, split)
        if df is not None and not df.empty:
            dfs.append(df)
    return dfs

def align_and_aggregate_epochs(dfs, metric):
    """
    Dado N DataFrames de distintos folds (con columna 'epoch'),
    alinea por epoch y devuelve (epochs, mean_array, std_array).
    """
    if not dfs:
        return None, None, None
    epoch_sets = [set(df['epoch'].values) for df in dfs
                  if 'epoch' in df.columns and metric in df.columns]
    if not epoch_sets:
        return None, None, None
    common_epochs = sorted(set.intersection(*epoch_sets))
    if not common_epochs:
        common_epochs = sorted(set.union(*epoch_sets))

    matrix = []
    for df in dfs:
        if 'epoch' not in df.columns or metric not in df.columns:
            continue
        indexed = df.set_index('epoch')[metric]
        vals = [indexed.get(e, float('nan')) for e in common_epochs]
        matrix.append(vals)

    if not matrix:
        return None, None, None

    matrix = np.array(matrix, dtype=float)
    mean = np.nanmean(matrix, axis=0)
    std  = np.nanstd(matrix, axis=0, ddof=0)
    return np.array(common_epochs), mean, std

def get_mean_best_epoch(val_dfs, rank_metric, ascending):
    """
    Para cada fold, encuentra la época de val óptima según rank_metric
    leyendo val_metrics.csv (serie completa).
    Devuelve (media, std) de esas épocas. Consistente con test_utils:
    epoch es índice 0-based.
    """
    best_epochs = []
    for vdf in val_dfs:
        if vdf is None or 'epoch' not in vdf.columns or rank_metric not in vdf.columns:
            continue
        idx = vdf[rank_metric].agg('idxmin' if ascending else 'idxmax')
        best_epochs.append(float(vdf.loc[idx, 'epoch']))
    if not best_epochs:
        return None, None
    return float(np.mean(best_epochs)), float(np.std(best_epochs, ddof=0))

print('Utilidades de curvas cargadas ✓')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CARGA DEL DATAFRAME MAESTRO  (usa test_utils.collect_all_runs_df)
# ─────────────────────────────────────────────────────────────────────────────
all_runs = collect_all_runs_df(
    experiments_path=RESULTS_PATH,
    rank_metric=RANK_METRIC,
    ascending=ASCENDING,
    task_filter=TASK_FILTER,
    metric_cols=METRIC_COLS,
    verbose=True,
)

print(f'Total válidos: {len(all_runs)}')
if not all_runs.empty:
    print(f'Tareas:  {sorted(all_runs["task"].unique())}')
    print(f'Modelos: {sorted(all_runs["model"].unique())}')
    print(f'MIL:     {sorted(all_runs["mil"].unique())}')
    print(f'Folds detectados (ejemplo primer run): {all_runs["n_folds"].iloc[0]}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS DE VISUALIZACIÓN
# ─────────────────────────────────────────────────────────────────────────────
DISPLAY_METRICS = METRIC_COLS
META_COLS = ['lr', 'wd', 'freeze', 'batch_size', 'optimizer', 'n_folds']


def fmt_mean_std(df, col):
    """Formatea col como 'mean ± std' solo si n_folds > 1 y existe col_std."""
    std_col = f'{col}_std'
    has_std = std_col in df.columns
    def _fmt(r):
        if pd.isna(r.get(col, float('nan'))):
            return '—'
        n = r.get('n_folds', 1)
        if has_std and n > 1 and pd.notna(r.get(std_col, float('nan'))):
            return f"{r[col]:.4f} ± {r[std_col]:.4f}"
        return f"{r[col]:.4f}"
    return df.apply(_fmt, axis=1)


def show_group_table(group_df, group_label, top_n):
    top = group_df.head(top_n).copy()
    top.index = range(1, len(top) + 1)

    extra = [c for c in ['best_epoch'] if c in top.columns]
    metric_cols = [c for c in DISPLAY_METRICS if c in top.columns]
    multi_fold = int(top['n_folds'].iloc[0]) > 1 if 'n_folds' in top.columns else False
    std_suffix = ' ± std' if multi_fold else ''

    display_df = top[[c for c in META_COLS + extra if c in top.columns]].copy()
    for col in metric_cols:
        label = f'► {col} (best_val){std_suffix}' if col == RANK_METRIC else f'{col}{std_suffix}'
        display_df[label] = fmt_mean_std(top, col)

    display(HTML(f"<h4 style='margin-top:16px; margin-bottom:2px'>{group_label}</h4>"))
    display(display_df.style.format({
        'lr': '{:.0e}',
        'wd': '{:.0e}',
    }, na_rep='—'))


def barplot_task(task_df, task):
    """Barplot con media y barras de error (std) entre folds."""
    candidates   = [RANK_METRIC] + [m for m in DISPLAY_METRICS if m != RANK_METRIC]
    plot_metrics = [m for m in candidates if m in task_df.columns and task_df[m].notna().any()]

    if not plot_metrics:
        print(f'Sin métricas disponibles para {task}')
        return

    models = sorted(task_df['model'].unique())
    mils   = sorted(task_df['mil'].unique())
    mil_cmap  = plt.get_cmap('tab10')
    mil_color = {m: mil_cmap(i) for i, m in enumerate(mils)}

    bar_w       = 0.8 / max(len(mils), 1)
    encoder_gap = 0.5

    # ── Mejor MIL por encoder según RANK_METRIC ──
    rank_std_col = f'{RANK_METRIC}_std'
    rank_agg_cols = {RANK_METRIC: ('min' if ASCENDING else 'max')}
    if rank_std_col in task_df.columns:
        rank_agg_cols[rank_std_col] = 'mean'
    rank_agg_df = task_df.groupby(['model', 'mil']).agg(rank_agg_cols).reset_index()
    rank_agg_df.columns = ['model', 'mil'] + list(rank_agg_cols.keys())

    best_mil_per_model = {}
    for model in models:
        sub = rank_agg_df[rank_agg_df['model'] == model]
        if sub.empty:
            continue
        best_idx = sub[RANK_METRIC].idxmin() if ASCENDING else sub[RANK_METRIC].idxmax()
        best_mil_per_model[model] = sub.loc[best_idx, 'mil']

    def make_fig(metric):
        std_col = f'{metric}_std'
        has_std = std_col in task_df.columns

        agg_cols = {metric: ('min' if ASCENDING else 'max')}
        if has_std:
            agg_cols[std_col] = 'mean'

        agg = task_df.groupby(['model', 'mil']).agg(agg_cols).reset_index()
        agg.columns = ['model', 'mil'] + list(agg_cols.keys())

        positions, group_centers, sep_lines = [], {}, []
        x = 0.0
        for ei, model in enumerate(models):
            block_start = x
            for mi, mil in enumerate(mils):
                row = agg[(agg['model'] == model) & (agg['mil'] == mil)]
                if row.empty:
                    continue
                val = float(row[metric].iloc[0])
                err = float(row[std_col].iloc[0]) if has_std else 0.0
                positions.append((x + mi * bar_w + bar_w / 2, val, err, mil, model))
            block_end = x + len(mils) * bar_w
            group_centers[model] = (block_start + block_end) / 2
            if ei < len(models) - 1:
                sep_lines.append(block_end + encoder_gap / 2)
            x = block_end + encoder_gap
        total_width = x - encoder_gap

        fig, ax = plt.subplots(figsize=(max(5, total_width * 1.4), 4.8))

        for (bx, val, err, mil, model) in positions:
            ax.bar(bx, val, width=bar_w * 0.88,
                   color=mil_color[mil], edgecolor='white', linewidth=0.5, zorder=2,
                   yerr=err if err > 0 else None,
                   capsize=4, error_kw=dict(elinewidth=1.2, capthick=1.2, ecolor='#333'))

        ax.autoscale(axis='y')
        ymin_auto, ymax_auto = ax.get_ylim()
        y_range = ymax_auto - ymin_auto

        for (bx, val, err, mil, model) in positions:
            if best_mil_per_model.get(model) != mil:
                continue
            label_str = f'{val:.2f}\n±{err:.2f}' if err > 0 else f'{val:.2f}'
            ax.text(bx, val + err + y_range * 0.004, label_str,
                    ha='center', va='bottom', fontsize=7,
                    linespacing=1.2, fontweight='bold')

        for xsep in sep_lines:
            ax.axvline(xsep, color='#aaaaaa', linewidth=1.0, linestyle='--', zorder=1)
        ax.set_xticks(list(group_centers.values()))
        ax.set_xticklabels(list(group_centers.keys()), fontsize=10, fontweight='bold')
        ax.tick_params(axis='x', length=0)
        ax.set_ylabel(f'{metric} (best_val)', fontsize=8)
        ax.set_title(f'{task}  —  {metric}  (media ± std entre folds)', fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
        ax.grid(axis='y', alpha=0.3, zorder=1)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlim(-0.4, total_width + 0.4)
        mil_handles = [plt.Rectangle((0,0),1,1, color=mil_color[m], label=m) for m in mils]
        ax.legend(handles=mil_handles, title='MIL', fontsize=8, title_fontsize=8, loc='lower right')
        plt.tight_layout()
        return fig

    import io, base64
    def fig_to_b64(fig):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        return base64.b64encode(buf.read()).decode()

    GROUP = 3
    for start in range(0, len(plot_metrics), GROUP):
        chunk = plot_metrics[start:start + GROUP]
        imgs_html = ''
        for metric in chunk:
            fig = make_fig(metric)
            b64 = fig_to_b64(fig)
            imgs_html += (
                f'<a href="data:image/png;base64,{b64}" download="{task}_{metric}.png" '
                f'title="Descargar {metric}">'
                f'<img src="data:image/png;base64,{b64}" '
                f'style="height:340px; margin:4px; border:1px solid #ddd; '
                f'border-radius:4px; cursor:pointer;" /></a>'
            )
        display(HTML(
            f'<div style="display:flex; flex-wrap:nowrap; overflow-x:auto; '
            f'gap:6px; padding:4px 0;">{imgs_html}</div>'
        ))


print('Helpers cargados ✓')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RANKING POR TAREA → encoder × MIL  +  barplot por tarea
# Muestra TOP_N experimentos por combinación encoder×MIL, con todos los
# hiperparámetros rankeados dentro de cada grupo.
# ─────────────────────────────────────────────────────────────────────────────
if all_runs.empty:
    print('⚠️  No se encontraron experimentos válidos. Revisa RESULTS_PATH y RANK_METRIC.')
else:
    display(HTML(
        f'<h2>Ranking — <code>{RANK_METRIC}</code> (best_val promediada)'
        f' | Top {TOP_N} por tarea × encoder × MIL<br>'
        f'<span style="font-size:0.75em; font-weight:normal; color:#555">'
        f'Las métricas muestran media ± std entre los k folds</span></h2>'
    ))

    for task in sorted(all_runs['task'].unique()):
        task_df = all_runs[all_runs['task'] == task]

        display(HTML(
            f"<h2 style='margin-top:40px; border-bottom:2px solid #555; padding-bottom:6px'>"
            f"📋 Tarea: {task} "
            f"<span style='font-size:0.7em; font-weight:normal; color:#666'>({len(task_df)} experimentos)</span></h2>"
        ))

        for (model, mil), grp in task_df.groupby(['model', 'mil'], sort=True):
            show_group_table(grp, f'🧪 encoder={model} | MIL={mil}  ({len(grp)} runs)', TOP_N)

        display(HTML(f"<h4 style='margin-top:28px'>📊 Comparativa visual — tarea: {task}</h4>"))
        barplot_task(task_df, task)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RANKING GLOBAL POR TAREA
# Muestra los TOP_N_GLOBAL mejores experimentos de cada tarea, con todos
# sus hiperparámetros y métricas.
# ─────────────────────────────────────────────────────────────────────────────
if not all_runs.empty:
    display(HTML(f'<h2>🏆 Ranking global por tarea — {RANK_METRIC} (best_val promediada)</h2>'))

    metric_cols_avail = [c for c in DISPLAY_METRICS if c in all_runs.columns]
    id_cols = ['model', 'mil', 'freeze', 'batch_size', 'lr', 'wd', 'optimizer', 'n_folds']

    for task in sorted(all_runs['task'].unique()):
        task_df = all_runs[all_runs['task'] == task].head(TOP_N_GLOBAL).copy()
        task_df.index = range(1, len(task_df) + 1)

        display(HTML(
            f"<h3 style='margin-top:32px; border-bottom:1px solid #888; padding-bottom:4px'>"
            f"📋 Tarea: {task} "
            f"<span style='font-size:0.75em; font-weight:normal; color:#666'>"
            f"(top {len(task_df)} de {len(all_runs[all_runs['task']==task])} experimentos)</span></h3>"
        ))

        display_df = task_df[[c for c in id_cols if c in task_df.columns]].copy()
        multi_fold = int(task_df['n_folds'].iloc[0]) > 1 if 'n_folds' in task_df.columns else False
        std_suffix = ' ± std' if multi_fold else ''
        for col in metric_cols_avail:
            label = f'► {col} (best_val){std_suffix}' if col == RANK_METRIC else f'{col}{std_suffix}'
            display_df[label] = fmt_mean_std(task_df, col)

        display(display_df.style.format(na_rep='—'))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CURVAS DE EVOLUCIÓN — PROMEDIO ENTRE FOLDS ± BANDA DE DESVIACIÓN ESTÁNDAR
#
# Decisión de diseño:
#   Una gráfica por métrica con curva promedio entre folds + banda ±1 std.
#   Línea vertical azul = época media en la que val fue óptimo entre los folds.
#   La época mostrada es consistente con la usada por test.py (índice 0-based).
# ─────────────────────────────────────────────────────────────────────────────

def plot_curves_kfold(run_name, label=None):
    run_dir   = os.path.join(RESULTS_PATH, run_name)
    try:
        fold_dirs = get_fold_dirs(run_dir)
    except ValueError as e:
        print(f'  ⚠️  {e}')
        return
    n_folds = len(fold_dirs)

    train_dfs = load_fold_epoch_dfs(run_dir, 'train')
    val_dfs   = load_fold_epoch_dfs(run_dir, 'val')

    available = [m for m in PLOT_METRICS if
                 any(m in df.columns for df in train_dfs + val_dfs)]

    if not available:
        print(f'  Sin métricas disponibles: {run_name}')
        return

    # Época media de mejor val (línea vertical)
    # Usa val_metrics.csv completo para calcular en qué época fue el óptimo,
    # coherente con lo que guarda best_eval.csv y usa test_utils.
    mean_best_ep, std_best_ep = get_mean_best_epoch(val_dfs, RANK_METRIC, ASCENDING)

    fig, axes = plt.subplots(1, len(available),
                             figsize=(5 * len(available), 4),
                             constrained_layout=True)
    if len(available) == 1:
        axes = [axes]

    colors = {'train': '#2196F3', 'val': '#FF9800'}

    for ax, metric in zip(axes, available):
        # ── Train (media ± std) ──────────────────────────────────────────
        eps_tr, mean_tr, std_tr = align_and_aggregate_epochs(train_dfs, metric)
        if eps_tr is not None:
            ax.plot(eps_tr, mean_tr, label=f'Train (n={len(train_dfs)})',
                    color=colors['train'], alpha=0.9, linewidth=1.8)
            if n_folds > 1:
                ax.fill_between(eps_tr, mean_tr - std_tr, mean_tr + std_tr,
                                color=colors['train'], alpha=0.15)

        # ── Val (media ± std) ────────────────────────────────────────────
        eps_v, mean_v, std_v = align_and_aggregate_epochs(val_dfs, metric)
        if eps_v is not None:
            ax.plot(eps_v, mean_v, label=f'Val (n={len(val_dfs)})',
                    color=colors['val'], alpha=0.9, linewidth=1.8)
            if n_folds > 1:
                ax.fill_between(eps_v, mean_v - std_v, mean_v + std_v,
                                color=colors['val'], alpha=0.15)

        # ── Línea vertical: época media de mejor val ──────────────────────
        if mean_best_ep is not None:
            ep_std_str = f' ±{std_best_ep:.1f}' if n_folds > 1 and std_best_ep > 0 else ''
            ax.axvline(mean_best_ep, linestyle='--', color='#5C6BC0', alpha=0.8, linewidth=1.5,
                       label=f'Best val ep {mean_best_ep:.1f}{ep_std_str}')

        x_max = int(max(
            (eps_tr[-1] if eps_tr is not None and len(eps_tr) else 0),
            (eps_v[-1]  if eps_v  is not None and len(eps_v)  else 0)
        ))
        ax.set_xticks(range(0, x_max + 1, 2))

        ax.set_title(metric, fontsize=9)
        ax.set_xlabel('Época (índice 0-based)')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

    suptitle = label or run_name
    if n_folds > 1:
        suptitle += f'\n(curvas = media ± std entre {n_folds} folds)'
    fig.suptitle(suptitle, fontsize=8, wrap=True)
    plt.show()


if not all_runs.empty:
    display(HTML(
        '<h2>📈 Curvas de evolución — mejor experimento por combinación encoder × MIL</h2>'
        '<p style="color:#555; font-size:0.9em;">'
        'Curva = <b>promedio entre folds</b>; banda sombreada = ±1 std.<br>'
        'Línea azul vertical = época media de mejor val entre folds (índice 0-based).<br>'
        'epochs_final = round(época_azul) + 1  →  lo que usará test.py.<br>'
        'Ordenados de mejor a peor por <code>RANK_METRIC</code>; el primero es el mejor global de la tarea.</p>'
    ))

    for task in sorted(all_runs['task'].unique()):
        task_df = all_runs[all_runs['task'] == task].dropna(subset=['rank_value'])
        if task_df.empty:
            continue

        display(HTML(
            f"<h3 style='margin-top:40px; border-bottom:2px solid #555; padding-bottom:6px'>"
            f"📋 Tarea: {task} "
            f"<span style='font-size:0.7em; font-weight:normal; color:#666'>({len(task_df)} experimentos)</span></h3>"
        ))

        # Mejor experimento por cada combinación encoder × MIL
        # (task_df ya viene ordenado por rank_value desc)
        seen_combos = {}
        for _, row in task_df.iterrows():
            key = (row['model'], row['mil'])
            if key not in seen_combos:
                seen_combos[key] = row

        best_per_combo = sorted(
            seen_combos.values(),
            key=lambda r: r['rank_value'],
            reverse=not ASCENDING
        )

        global_best_run = best_per_combo[0]['run_name'] if best_per_combo else None

        for rank_i, row in enumerate(best_per_combo):
            run_name = row['run_name']
            rv       = row['rank_value']
            rv_std   = row.get(f'{RANK_METRIC}_std', float('nan'))
            std_str  = f' ±{rv_std:.4f}' if pd.notna(rv_std) and row.get('n_folds', 1) > 1 else ''
            lr_val   = row.get('lr', float('nan'))
            wd_val   = row.get('wd', float('nan'))
            model    = row.get('model', '?')
            mil      = row.get('mil', '?')
            # best_epoch: media de las mejores épocas entre folds (0-based)
            best_ep  = row.get('best_epoch', None)
            ep_str   = f'  |  best_epoch: {best_ep:.1f}  →  epochs_final={int(round(best_ep))+1}' if best_ep is not None else ''

            is_best = (run_name == global_best_run)
            border  = '3px solid #FFD700' if is_best else '1px solid #ccc'
            bg      = '#fffdf0' if is_best else '#fafafa'

            title_str = (
                f"#{rank_i + 1}  Tarea: {task}  |  MIL: {mil}  |  Encoder: {model}"
                f"  |  lr: {lr_val:.0e}  |  wd: {wd_val:.0e}"
                f"  —  {RANK_METRIC}: {rv:.4f}{std_str}{ep_str}"
            )

            display(HTML(
                f"<div style='margin-top:24px; padding:6px 12px; "
                f"border-left:{border}; background:{bg}; border-radius:4px;'>"
                f"<b style='font-size:0.95em; color:#111;'>{title_str}</b></div>"
            ))

            plot_curves_kfold(run_name, label=title_str)

    # ── Experimento extra opcional ────────────────────────────────────────────
    if EXTRA_RUN is not None:
        display(HTML(
            "<h2 style='margin-top:40px; border-bottom:2px solid #e07b00; padding-bottom:4px'>"
            "🔍 Experimento extra</h2>"
        ))
        print(f'Run: {EXTRA_RUN}')
        plot_curves_kfold(EXTRA_RUN)
